# Living LLM text-watermark evidence audit
This notebook validates the strict admitted-evidence ledger, then summarizes the broader discovery bibliography and creates update queues. It does not use citation count as a quality score.

In [ ]:
from datetime import date
import json
from pathlib import Path
from jsonschema import Draft202012Validator, FormatChecker
import polars as pl


def find_repo_root(start=Path.cwd()):
    """Find the checkout root from any working directory inside the repository."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'research').is_dir():
            return candidate
    raise FileNotFoundError('Repository root not found; launch from within the checkout.')


REPO_ROOT = find_repo_root()
LEDGER = REPO_ROOT / 'research' / 'frontier_evidence.json'
LEDGER_SCHEMA = REPO_ROOT / 'schemas' / 'literature-evidence.schema.json'
ledger = json.loads(LEDGER.read_text(encoding='utf-8'))
ledger_schema = json.loads(LEDGER_SCHEMA.read_text(encoding='utf-8'))
ledger_errors = list(Draft202012Validator(ledger_schema, format_checker=FormatChecker()).iter_errors(ledger))
assert not ledger_errors, [error.message for error in ledger_errors]
record_ids = [record['record_id'] for record in ledger['records']]
assert len(record_ids) == len(set(record_ids)), 'Duplicate admitted record_id values'
assert all(record['index_records'] and record['pdf_url'].startswith('https://') for record in ledger['records'])
print(f'{len(record_ids)} admitted records validated from {LEDGER}')

CSV = REPO_ROOT / 'research' / 'annotated_bibliography.csv'
bib = pl.read_csv(CSV, infer_schema_length=1000)
print(f'{bib.height} bibliography records loaded from {CSV}')
bib.head()

In [ ]:
required = {'record_id', 'year', 'title', 'status', 'method_family', 'evidence_role', 'url', 'annotation', 'version_checked', 'checked_at'}
missing_columns = required - set(bib.columns)
assert not missing_columns, f'Missing columns: {sorted(missing_columns)}'
assert bib.select(pl.col('record_id').n_unique()).item() == bib.height, 'Duplicate record_id values'
duplicate_doi = (bib.filter(pl.col('doi').is_not_null() & (pl.col('doi') != ''))
                 .group_by('doi').len().filter(pl.col('len') > 1))
duplicate_arxiv = (bib.filter(pl.col('arxiv').is_not_null() & (pl.col('arxiv') != ''))
                    .group_by('arxiv').len().filter(pl.col('len') > 1))
print('Duplicate DOI rows:', duplicate_doi.height)
print('Duplicate arXiv rows:', duplicate_arxiv.height)

In [ ]:
coverage = (bib.group_by(['evidence_role', 'status'])
            .len().sort(['evidence_role', 'status']))
coverage

In [ ]:
today = date.today()
update_queue = (bib.with_columns(pl.col('checked_at').str.to_date().alias('checked_date'))
    .with_columns((pl.lit(today) - pl.col('checked_date')).dt.total_days().alias('days_since_check'))
    .filter((pl.col('status') == 'preprint') | (pl.col('days_since_check') > 180))
    .select('record_id', 'title', 'status', 'version_checked', 'checked_at', 'days_since_check', 'url')
    .sort('days_since_check', descending=True))
update_queue

## Research-agent update actions
For every queued record, search Google Scholar, Semantic Scholar, Crossref, OpenAlex, DBLP, and the official venue. Verify all substantive changes in the primary PDF. Update the discovery CSV, then admit the record to `frontier_evidence.json` only if it has a clear citation, a scholarly index record, and a directly downloadable PDF. Preserve prior versions and run this notebook again.

In [ ]:
missing_identifiers = bib.filter(
    ((pl.col('doi').is_null()) | (pl.col('doi') == '')) &
    ((pl.col('arxiv').is_null()) | (pl.col('arxiv') == ''))
).select('record_id', 'title', 'venue', 'url')
missing_identifiers